# Aidge Graph Compiler Tutorial

This simple notebook uses a specific **Aidge exporter** named **`aidge_export_compile`** in order to export an aidge graph.

Actually the notebook uses several repositories in addition to Aidge, in particular:
- Aidge: `aidge_core`, `aidge_expport_cpp`
- Aidge compiler export: `aidge_export_compile`
- Compiler frameworks: `XTC` and `tvm` or `mlir`
- Network test and evaluation harness: `aidge_benchmark`

By the end of this notebook, you will have seen how the default Aidge C++ export and the Aidge Compiler generate
different build system files for the operators in the graph.

## Export and Build a Graph withthe C++ export

There we simply use the `aidge_nn` repository which basically wraps calls to either the default Aidge C++ exporter or
the AIdge Compiler Exporter.

**Practice** Use the `params` namespace to vary parameters, for instance:
- `model`: can be `resnet8` or `resnet18` for instance
- `compile`: can be set to True for enabling the compiler
- `comp_modules`: can be set to `csrc` to generate optimized C sources, or `arlib` to generate commpiled archives directly
- `comp_level`: can be between 0 and 4, where 4 uses autotuning database
  
Execute the cells, the timing (seconds for inference) is outputed raw in the console.

Can you observe the performance difference between C++ export and Compiled export?

**Warning** Due to a bug currently in the notebook, always rerun the full notebook with the `>>` button above!

In [1]:
from argparse import Namespace as NS              
import subprocess as sp      
import logging

import aidge_core     
   
from aidge_nn.testing.models import (
    get_config,
    get_model,
    get_input,
    export_model,
)
from aidge_nn.testing.utils.workload import initialize_producers

logger = logging.getLogger("notebook")

# Vary some parameters there
params = NS(
    model="resnet8",
    compile=False,
    comp_level=3,
    comp_module="arlib",
    debug=False,
    batch=1,
    randomize=True,
    seed=42,
    get_time=True,
    repeat=1,
)
args = get_config(params)

logging.basicConfig()
logger.setLevel(logging.INFO)
if args.debug:
    logging.getLogger("aidge_export_compile").setLevel(logging.DEBUG)
else:
    logging.getLogger("aidge_export_compile").setLevel(logging.INFO)
if args.debug_aidge:
    aidge_core.Log.set_console_level(aidge_core.Level.Debug)
else:
    aidge_core.Log.set_console_level(aidge_core.Level.Error)

# Get specified model
config = get_model(args)

# Create an input with specified batch
config = get_input(config, args)

# Compile model graph
config.model.compile(
    "cpu",
    getattr(aidge_core.dtype, config.input_dtype),
    dims=[config.input_shape]
)

# Randomize parameters
initialize_producers(config.model, randomize=args.randomize, seed=args.seed)

# Export (optionally compile)
logger.info("Export model to: %s/", args.export_dir)
export_model(config, args.export_dir, args)

sp.run(f"make -C {args.export_dir} CC='g++'", shell=True)
sp.run(f"{args.export_dir}/bin/run_export", shell=True)


INFO:notebook:Export model to: export/


gen : export/data/bn_data_input_0.h
make: Entering directory '/home/cguillon/work/xtc-tutorial-cpsiot/xtc-aidge-tutorial/export'
g++  -O2 -Wall -Wextra -Wno-unused-function -MMD -fopenmp -I. -I./dnn -I./dnn/include -I./dnn/layers -I./dnn/parameters -c dnn/src/parameters/conv0_w.cpp -o build/./dnn/src/parameters/conv0_w.o 
g++  -O2 -Wall -Wextra -Wno-unused-function -MMD -fopenmp -I. -I./dnn -I./dnn/include -I./dnn/layers -I./dnn/parameters -c dnn/src/parameters/stage1_unit1_bn1_shift.cpp -o build/./dnn/src/parameters/stage1_unit1_bn1_shift.o 
g++  -O2 -Wall -Wextra -Wno-unused-function -MMD -fopenmp -I. -I./dnn -I./dnn/include -I./dnn/layers -I./dnn/parameters -c dnn/src/parameters/stage3_unit1_conv1_w.cpp -o build/./dnn/src/parameters/stage3_unit1_conv1_w.o 
g++  -O2 -Wall -Wextra -Wno-unused-function -MMD -fopenmp -I. -I./dnn -I./dnn/include -I./dnn/layers -I./dnn/parameters -c dnn/src/parameters/stage2_unit1_conv2_b.cpp -o build/./dnn/src/parameters/stage2_unit1_conv2_b.o 
g++  -O2 

CompletedProcess(args='export/bin/run_export', returncode=0)

### Inspect generated file

One may Inspect the generated tree below and optionally get into the notebook files to inspect the files.


In [2]:
!tree export/dnn

export/dnn
├── include
│   ├── forward.hpp
│   ├── kernels
│   │   ├── activation.hpp
│   │   ├── batchnorm.hpp
│   │   ├── convolution.hpp
│   │   ├── elemwise.hpp
│   │   ├── fullyconnected.hpp
│   │   └── pooling.hpp
│   ├── layers
│   │   ├── bn1.h
│   │   ├── bn_data.h
│   │   ├── conv0.h
│   │   ├── fc.h
│   │   ├── global_avg_pool.h
│   │   ├── relu1.h
│   │   ├── stage1_unit1_add.h
│   │   ├── stage1_unit1_bn1.h
│   │   ├── stage1_unit1_bn2.h
│   │   ├── stage1_unit1_conv1.h
│   │   ├── stage1_unit1_conv2.h
│   │   ├── stage1_unit1_relu1.h
│   │   ├── stage1_unit1_relu2.h
│   │   ├── stage1_unit1_shortcut.h
│   │   ├── stage2_unit1_add.h
│   │   ├── stage2_unit1_bn1.h
│   │   ├── stage2_unit1_bn2.h
│   │   ├── stage2_unit1_conv1.h
│   │   ├── stage2_unit1_conv2.h
│   │   ├── stage2_unit1_relu1.h
│   │   ├── stage2_unit1_relu2.h
│   │   ├── stage2_unit1_shortcut.h
│   │   ├── stage3_unit1_add.h
│   │   ├── stage3_unit1_bn1.h
│   │   ├── stage3_unit1_bn2.h
│   │   ├── stage3_unit